# 📬 Pulso Académico SIIMAG — v4

Boletín trimestral con diseño editorial. Colores institucionales `#0033A0` y blanco, logo oficial SIIMAG.

**Novedades v4:**
- Texto introductorio con estilo periodístico tipo Metro: directo, con doble sentido suave
- Asunto del correo con gancho periodístico generado por Gemini
- Semáforo correcto por categoría (bajas: rojo si sube, verde si baja)

**Compatible con:** Gmail (Google Workspace) ✅

In [1]:
# ── 1. SETUP ────────────────────────────────────────────────────────────────
!pip install git+https://github.com/claudiodanielpc-ag/cd_base.git -q
!pip install unidecode -q
!pip uninstall -y paramiko sshtunnel
!pip install paramiko==2.11.0 sshtunnel==0.4.0

from cd_base import ConexionBD
from google.colab import drive, ai
drive.mount('/content/drive')

import pandas as pd
import re, unidecode, smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

print('✅ Librerías listas')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Found existing installation: paramiko 2.11.0
Uninstalling paramiko-2.11.0:
  Successfully uninstalled paramiko-2.11.0
Found existing installation: sshtunnel 0.4.0
Uninstalling sshtunnel-0.4.0:
  Successfully uninstalled sshtunnel-0.4.0
  Using cached paramiko-2.11.0-py2.py3-none-any.whl.metadata (4.6 kB)
  Using cached sshtunnel-0.4.0-py2.py3-none-any.whl.metadata (19 kB)
Using cached paramiko-2.11.0-py2.py3-none-any.whl (212 kB)
Using cached sshtunnel-0.4.0-py2.py3-none-any.whl (24 kB)


/usr/local/lib/python3.12/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/local/lib/python3.12/dist-packages/paramiko/transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Librerías listas


In [2]:
# ── 2. CONEXIÓN A BD ─────────────────────────────────────────────────────────
ruta   = '/content/drive/MyDrive/credenciales/bd_produccion.txt'
bd     = ConexionBD(ruta)
engine = bd.conectar('migracion_aws_do')

✅ Conectado a base: migracion_aws_do


In [3]:
# ── 3. PARÁMETROS DEL PERIODO ─────────────────────────────────────────────────
PERIODO_ACTUAL_INI = '2026-01-01'
PERIODO_ACTUAL_FIN = '2026-04-30'
PERIODO_COMP_INI   = '2025-01-01'
PERIODO_COMP_FIN   = '2025-04-30'
PERIODO_LABEL      = 'Enero – Abril 2026'
PERIODO_COMP_LABEL = 'Enero – Abril 2025'
PERIODO_CORTO      = 'ENE–ABR 2026'
ANIO_COMP          = '2025'

LOGO_URL = 'https://raw.githubusercontent.com/claudiodanielpc-ag/siimag/refs/heads/main/logo_siimag_nuevo_transp.png'

In [4]:
# ── 4. EXTRACCIÓN DE INDICADORES ─────────────────────────────────────────────
conn   = engine.raw_connection()
cursor = conn.cursor()

cursor.callproc(
    'sp_tbl_indicadores_general_programas',
    (1, None, None,
     PERIODO_ACTUAL_INI, PERIODO_ACTUAL_FIN,
     PERIODO_COMP_INI,   PERIODO_COMP_FIN)
)

resultados = []
while True:
    rows = cursor.fetchall()
    if rows:
        cols = [col[0] for col in cursor.description]
        resultados.append(pd.DataFrame(rows, columns=cols))
    if not cursor.nextset():
        break
cursor.close()

indicadores = resultados[0]

# Filtrar solo los indicadores que se mostrarán en este boletín
keep_indicadores = [
    'Inscritos',
    'Inscritos facturados',
    'Cargas',
    'Eficiencia terminal',
    'Tasa de Deserción'
]
indicadores = indicadores[indicadores['nombre'].isin(keep_indicadores)].copy()
indicadores['nombre'] = indicadores['nombre'].replace({'Cargas': 'Cargas de materias'})
orden = {
    'Inscritos': 0,
    'Inscritos facturados': 1,
    'Cargas de materias': 2,
    'Eficiencia terminal': 3,
    'Tasa de Deserción': 4,
}
indicadores['orden'] = indicadores['nombre'].map(orden)
indicadores = indicadores.sort_values('orden').reset_index(drop=True)
indicadores.drop(columns=['orden'], inplace=True)

print(f'✅ {len(indicadores)} indicadores cargados')
indicadores

✅ 5 indicadores cargados


,id_indicador,nombre,porcentual,categoria,total,total_comp,porcentaje,color_sube,color_baja,descripcion,formula,tendencia
0,3,Inscritos,0,1,884.00,1010.000000,-12.48,#388e3c,#f44336,Alumnos dados de alta en el programa académico.,None,down
1,4,Inscritos facturados,0,1,731.00,804.000000,-9.08,#388e3c,#f44336,Alumnos inscritos que ya han cargado al menos ...,None,down
2,6,Cargas de materias,0,1,21074.00,22212.000000,-5.12,#388e3c,#f44336,Materias que los alumnos inscriben durante un ...,None,down
3,9,Eficiencia terminal,1,1,30.37,23.969999,26.70,#388e3c,#f44336,Porcentaje de estudiantes que ya concluyeron ...,Alumnos que Finalizaron Materias / Alumnos que...,up
4,8,Tasa de Deserción,1,3,37.89,29.230000,29.63,#f44336,#388e3c,Porcentaje de estudiantes que han decidido no ...,Bajas de programas/ inscritos facturados,up


In [5]:
# ── 4-ALT. DATOS DE EJEMPLO (descomenta si no tienes BD) ─────────────────────

# indicadores = pd.DataFrame([
#   {'id_indicador':3,  'nombre':'Inscritos',            'porcentual':0,'categoria':1,'total':570,  'total_comp':641,   'porcentaje':-11.08,'tendencia':'down','descripcion':'Alumnos dados de alta en el programa académico.'},
#   {'id_indicador':4,  'nombre':'Inscritos facturados', 'porcentual':0,'categoria':1,'total':558,  'total_comp':542,   'porcentaje': 2.95, 'tendencia':'up',  'descripcion':'Inscritos con al menos una materia cargada.'},
#   {'id_indicador':6,  'nombre':'Cargas',               'porcentual':0,'categoria':1,'total':15854,'total_comp':16854, 'porcentaje':-5.93, 'tendencia':'down','descripcion':'Materias inscritas durante el periodo.'},
#   {'id_indicador':9,  'nombre':'Eficiencia terminal',  'porcentual':1,'categoria':1,'total':30.37,'total_comp':25.51, 'porcentaje':19.05, 'tendencia':'up',  'descripcion':'Porcentaje de alumnos que concluyeron materias.'},
#   {'id_indicador':5,  'nombre':'Reactivados',          'porcentual':0,'categoria':2,'total':137,  'total_comp':321,   'porcentaje':-57.32,'tendencia':'down','descripcion':'Alumnos que retomaron su trayectoria.'},
#   {'id_indicador':12, 'nombre':'Tasa de reactivación', 'porcentual':1,'categoria':2,'total':63.72,'total_comp':126.88,'porcentaje':-49.78,'tendencia':'down','descripcion':'Porcentaje de alumnos que retoman vs. bajas.'},
#   {'id_indicador':10, 'nombre':'Reinscritos',          'porcentual':0,'categoria':2,'total':158,  'total_comp':160,   'porcentaje':-1.25, 'tendencia':'down','descripcion':'Alumnos que retomaron tras baja de empresa.'},
#   {'id_indicador':7,  'nombre':'Tasa de reinserción',  'porcentual':1,'categoria':2,'total':31.35,'total_comp':30.36, 'porcentaje': 3.26, 'tendencia':'up',  'descripcion':'Porcentaje que vuelve a inscribirse.'},
#   {'id_indicador':1,  'nombre':'Bajas de la empresa',  'porcentual':0,'categoria':3,'total':504,  'total_comp':527,   'porcentaje':-4.36, 'tendencia':'down','descripcion':'Alumnos que dejaron de laborar en la empresa.'},
#   {'id_indicador':2,  'nombre':'Bajas del programa',   'porcentual':0,'categoria':3,'total':202,  'total_comp':187,   'porcentaje': 8.02, 'tendencia':'up',  'descripcion':'Alumnos que dejaron el programa pero siguen en empresa.'},
#   {'id_indicador':8,  'nombre':'Tasa de Deserción',    'porcentual':1,'categoria':3,'total':36.20,'total_comp':34.50, 'porcentaje': 4.93, 'tendencia':'up',  'descripcion':'Porcentaje que no continúa el programa.'},
# ])
# print('✅ Datos de ejemplo listos')

In [6]:
# ── 5. TEXTOS CON GEMINI — ESTILO METRO ──────────────────────────────────────
#
# El periódico Metro (CDMX) es famoso por titulares cortos con doble sentido,
# juego de palabras y frases que te detienen aunque vayas volando.
# Aquí le pedimos a Gemini que escriba con ese espíritu: directo, chispeante
# y con un guiño de doble sentido — sin pasarse de la raya para el contexto
# corporativo/académico.

resumen = indicadores[['nombre','total','total_comp','porcentaje']].to_string(index=False)

# ── 5a. Texto introductorio ───────────────────────────────────────────────────
prompt_intro = f"""
Eres redactor del periódico Metro de la Ciudad de México. Tu estilo es directo,
chispeante y usas doble sentido suave — palabras que en el mundo académico y
corporativo se lean con picardía sin resultar ofensivas. Ejemplos del Metro:
"Más alumnos se apuntan", "La deserción no suelta", "Se les fue la mano con las bajas",
"Los reactivados... no tan activos", "La eficiencia terminal: terminaron bien".

Escribe un párrafo introductorio de MÁXIMO 3 oraciones para el boletín trimestral
del SIIMAG (Academia Global) del periodo {PERIODO_LABEL}.
- Menciona primero 1 o 2 datos clave del periodo actual y luego un breve resumen
  comparativo contra {PERIODO_COMP_LABEL}.
- Menciona 2 o 3 indicadores con sus valores numéricos.
- Usa al menos un juego de palabras o doble sentido suave relacionado con
  el mundo académico (inscribirse, cargar materias, darse de baja, terminal,
  reactivar, desertar, reinsertar, eficiencia).
- Tono: ingenioso pero profesional. Sin groserías. Sin listas. Solo prosa.
- No pongas título ni encabezado. No uses asteriscos ni markdown.

Datos del periodo vs. {PERIODO_COMP_LABEL}:
{resumen}
"""

texto_intro = ai.generate_text(prompt_intro).strip()

# ── 5b. Asunto del correo ─────────────────────────────────────────────────────
prompt_asunto = f"""
Eres redactor del periódico Metro CDMX. Escribe UN SOLO asunto de correo
(máximo 10 palabras) para el boletín del SIIMAG del periodo {PERIODO_LABEL}.
- Usa doble sentido suave o juego de palabras académico.
- Que genere curiosidad y ganas de abrir el correo.
- Sin groserías. Sin emojis. Solo texto.
- Ejemplos del estilo deseado: "Unos se van, otros terminan bien",
  "La eficiencia subió... la deserción también quiso",
  "Más cargas, menos bajas: el trimestre en números".

Datos clave:
{resumen}
"""

asunto_correo = ai.generate_text(prompt_asunto).strip()

print('📝 INTRO:')
print(texto_intro)
print()

print('📨 ASUNTO:')
print(asunto_correo)

📝 INTRO:
El primer trimestre del año para el SIIMAG, de Enero a Abril 2026, nos dejó números que invitan a levantar la ceja, pues si bien la deserción no nos suelta y se disparó al 37.89% respecto al periodo anterior, los 884 inscritos también nos muestran una ligera contracción. Sin embargo, no todo fue ‘bajón’, ya que quienes lograron la terminal lo hicieron con creces, llevando la eficiencia a un brillante 30.37%, superando por mucho el año pasado.

📨 ASUNTO:
Menos ingresan, más concluyen... ¿la selección natural?


In [7]:
# ── 6. FUNCIONES AUXILIARES ───────────────────────────────────────────────────

CATEGORIAS = {1: 'Captación', 2: 'Retención', 3: 'Bajas y deserción'}

AZUL      = '#0033A0'
FONDO     = '#f5f7fa'
SEP       = '#e8edf5'
VERDE     = '#388E3C'
VERDE_NUM = '#1a6b1a'
ROJO      = '#D32F2F'
GRIS      = '#AAA'
GRIS_NUM  = '#555'
BG_VERDE  = '#E8F5E9'
TX_VERDE  = '#2E7D32'
BG_ROJO   = '#FFEBEE'
TX_ROJO   = '#C62828'

def fmt_valor(row):
    v = row['total']
    if row['porcentual'] == 1:
        return f"{v:,.1f}", '%'
    elif v >= 1000:
        return f"{int(v):,}", ''
    return f"{int(v)}", ''


def fmt_comp(row):
    v = row['total_comp']
    if row['porcentual'] == 1:
        return f"{v:,.1f}%"
    elif v >= 1000:
        return f"{int(v):,}"
    return f"{int(v)}"


def es_bueno(row):
    """
    Captación (1) y Retención (2): subir = bueno.
    Bajas (3): bajar = bueno (lógica invertida).
    """
    sube = row['porcentaje'] > 0
    return (not sube) if row['categoria'] == 3 else sube


def color_num(row):
    if row['porcentaje'] == 0: return GRIS_NUM
    return VERDE_NUM if es_bueno(row) else ROJO


def color_barra(row):
    if row['porcentaje'] == 0: return GRIS
    return VERDE if es_bueno(row) else ROJO


def badge_html(row):
    pct  = abs(row['porcentaje'])
    flec = '&#9650;' if row['porcentaje'] > 0 else '&#9660;'
    return (f'<span style="display:inline-block;background:rgba(0,51,160,0.08);'
            f'color:{AZUL};padding:4px 10px;border-radius:5px;font-size:10px;'
            f'font-family:monospace;font-weight:700;">{flec} {pct:.1f}%</span>')

print('✅ Funciones listas')


✅ Funciones listas


In [8]:
# ── 7. ÍCONOS SVG POR INDICADOR ───────────────────────────────────────────────
ICONOS = {
    3:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><rect x="2" y="2" width="8" height="8" rx="1" fill="{AZUL}" opacity="0.9"/><rect x="14" y="2" width="8" height="8" rx="1" fill="{AZUL}" opacity="0.5"/><rect x="2" y="14" width="8" height="8" rx="1" fill="{AZUL}" opacity="0.5"/><rect x="14" y="14" width="8" height="8" rx="1" fill="{AZUL}" opacity="0.2"/></svg>',
    4:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><circle cx="12" cy="12" r="10" stroke="{AZUL}" stroke-width="1.5" opacity="0.3"/><circle cx="12" cy="12" r="6" fill="{AZUL}" opacity="0.6"/></svg>',
    6:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><rect x="2" y="14" width="4" height="8" rx="1" fill="{AZUL}" opacity="0.9"/><rect x="8" y="9" width="4" height="13" rx="1" fill="{AZUL}" opacity="0.6"/><rect x="14" y="5" width="4" height="17" rx="1" fill="{AZUL}" opacity="0.4"/><rect x="20" y="2" width="4" height="20" rx="1" fill="{AZUL}" opacity="0.2"/></svg>',
    9:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><polygon points="12,3 22,21 2,21" stroke="{AZUL}" stroke-width="1.5" stroke-linejoin="round" fill="rgba(0,51,160,0.08)"/></svg>',
    5:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><path d="M12 2 L22 12 L12 22 L2 12 Z" stroke="{AZUL}" stroke-width="1.5" fill="rgba(0,51,160,0.06)"/><circle cx="12" cy="12" r="2" fill="{AZUL}"/></svg>',
    12: f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><circle cx="12" cy="12" r="10" stroke="{AZUL}" stroke-width="1.5" opacity="0.3"/><path d="M6 12 Q12 6 18 12" stroke="{AZUL}" stroke-width="1.5" fill="none" stroke-linecap="round"/></svg>',
    10: f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><rect x="2" y="2" width="20" height="20" rx="3" stroke="{AZUL}" stroke-width="1.5" fill="none" opacity="0.4"/><line x1="7" y1="12" x2="17" y2="12" stroke="{AZUL}" stroke-width="1.5"/></svg>',
    7:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><path d="M4 20 C4 12 20 12 20 4" stroke="{AZUL}" stroke-width="2" stroke-linecap="round" fill="none" opacity="0.6"/><circle cx="20" cy="4" r="3" fill="{AZUL}" opacity="0.8"/></svg>',
    1:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><path d="M12 3 L3 21 L21 21 Z" stroke="{AZUL}" stroke-width="1.5" stroke-linejoin="round" fill="rgba(0,51,160,0.08)"/></svg>',
    2:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><path d="M12 21 L3 3 L21 3 Z" stroke="{AZUL}" stroke-width="1.5" stroke-linejoin="round" fill="rgba(0,51,160,0.08)"/></svg>',
    8:  f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><path d="M4 20 C4 12 20 12 20 4" stroke="{AZUL}" stroke-width="2" stroke-linecap="round" fill="none"/><circle cx="20" cy="4" r="3" fill="{AZUL}"/></svg>',
}
DEFAULT_ICONO = f'<svg width="20" height="20" viewBox="0 0 24 24" fill="none"><circle cx="12" cy="12" r="9" stroke="{AZUL}" stroke-width="1.5" opacity="0.4"/></svg>'

print('✅ Íconos listos')

✅ Íconos listos


In [9]:
# ── 8. GENERADORES DE BLOQUES HTML ────────────────────────────────────────────

F  = "font-family:'Century Gothic',Arial,sans-serif;"
FM = "font-family:monospace;"

def bloque_grande(row):
    valor, sufijo = fmt_valor(row)
    badg  = badge_html(row)
    sup   = f'<sup style="font-size:13px;color:#90A4D4;font-weight:400;">{sufijo}</sup>' if sufijo else ''
    return f"""
    <td width="33%" style="background:#ffffff;padding:20px 18px;vertical-align:top;
                   border:1px solid {SEP};border-radius:16px;">
      <div style="font-size:28px;font-weight:800;line-height:1;margin-bottom:6px;
                  color:{AZUL};{F}">{valor}{sup}</div>
      <div style="font-size:10px;letter-spacing:1.5px;color:{AZUL};
                  text-transform:uppercase;margin-bottom:12px;font-weight:700;{F}">{row['nombre']}</div>
      <div style="font-size:10px;color:{AZUL};{FM}">{badg}</div>
    </td>"""

def bloque_horizontal(row):
    valor, sufijo = fmt_valor(row)
    comp  = fmt_comp(row)
    cn    = color_num(row)
    cb    = color_barra(row)
    sube  = row['porcentaje'] > 0
    flec  = '&#9650;' if sube else '&#9660;'
    pct   = abs(row['porcentaje'])
    cv    = VERDE if es_bueno(row) else ROJO
    sup   = f'<span style="font-size:12px;color:#CCC;">{sufijo}</span>' if sufijo else ''
    return f"""
    <td width="50%" style="background:#ffffff;padding:16px;vertical-align:top;">
      <table cellpadding="0" cellspacing="0" width="100%">
        <tr>
          <td width="3" style="background:{cb};border-radius:2px;">&nbsp;</td>
          <td style="padding-left:12px;">
            <div style="font-size:22px;font-weight:800;line-height:1;
                        margin-bottom:2px;color:{cn};{F}">{valor}{sup}</div>
            <div style="font-size:10px;letter-spacing:1px;color:#333;
                        text-transform:uppercase;margin-bottom:6px;font-weight:700;{F}">{row['nombre']}</div>
            <div style="font-size:10px;color:{cv};{FM}">{flec} {pct:.1f}%</div>
            <div style="font-size:10px;color:#CCC;margin-top:2px;{F}">vs. {comp} en {ANIO_COMP}</div>
          </td>
        </tr>
      </table>
    </td>"""

def etiqueta_seccion(label):
    return f"""
    <tr>
      <td colspan="99" style="padding:16px 24px 6px;">
        <table cellpadding="0" cellspacing="0" width="100%">
          <tr>
            <td style="font-size:9px;letter-spacing:3px;color:{AZUL};opacity:0.6;
                       text-transform:uppercase;font-weight:500;
                       padding-right:12px;white-space:nowrap;{F}">{label}</td>
            <td style="border-top:1px solid {SEP};width:100%;"></td>
          </tr>
        </table>
      </td>
    </tr>"""

def grilla(filas_html):
    return f"""
    <tr>
      <td colspan="99" style="padding:0 24px 4px;">
        <table width="100%" cellpadding="0" cellspacing="1"
          style="background:{SEP};border-radius:10px;overflow:hidden;">
          {filas_html}
        </table>
      </td>
    </tr>"""

def texto_categoria(label):
    mensajes = {
        'Captación': 'Este bloque resume la captación y la actividad de ingreso del trimestre.',
        'Retención': 'Estos indicadores muestran cómo se sostiene la matrícula y el avance de los alumnos.',
        'Bajas y deserción': 'A continuación se presentan las salidas del programa y la tasa de deserción.'
    }
    texto = mensajes.get(label)
    if not texto:
        return ''
    return f"""
    <tr>
      <td colspan="99" style="padding:0 24px 12px;">
        <div style="font-size:11px;color:#555;line-height:1.6;{F}">{texto}</div>
      </td>
    </tr>"""


In [10]:
texto_intro = """
Durante el período enero-abril de 2026, los principales indicadores de desempeño tuvieron el siguiente comportamiento con respecto al mismo período de 2025:

<ul style="padding-left:20px; margin-top:10px;">

<li>El número de <b>inscritos</b> disminuyó en 12.5% al pasar de 1,010 a 884.</li>

<li>Las <b>cargas de materias</b> registraron una reducción de 5.1% al disminuir de 22,212 a 21,074.</li>

<li>La <b>eficiencia terminal</b> tuvo un aumento de 6.4 puntos porcentuales, pasando del 24.0% a 30.4%.</li>

<li>Finalmente, la <b>tasa de deserción</b> incrementó en 8.7 puntos porcentuales al pasar de 29.2% a 37.9%.</li>

</ul>
"""

In [11]:
# ── 9. CONSTRUCCIÓN DEL HTML ──────────────────────────────────────────────────

filas_indicadores = ''
for i in range(0, len(indicadores), 3):
    par = indicadores.iloc[i:i+3]
    celdas = ''.join(bloque_grande(r) for _, r in par.iterrows())
    if len(par) < 3:
        celdas += ''.join(
            '<td width="33%" style="background:#fff;padding:20px 14px;vertical-align:top;"></td>'
            for _ in range(3 - len(par))
        )
    filas_indicadores += f'<tr>{celdas}</tr>'

html = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width,initial-scale=1.0">
  <title>Pulso Académico SIIMAG · {PERIODO_LABEL}</title>
</head>
<body style="margin:0;padding:0;background:{FONDO};{F}">

<table width="100%" cellpadding="0" cellspacing="0"
  style="background:{FONDO};padding:20px 0;">
<tr><td align="center">

<table width="620" cellpadding="0" cellspacing="0"
  style="background:#ffffff;max-width:620px;
         border-radius:12px;overflow:hidden;border:1px solid {SEP};">

  <!-- BARRA SUPERIOR -->
  <tr>
    <td colspan="99" style="padding:16px 24px 14px;border-bottom:1px solid {SEP};">
      <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
          <td>
            <img src="{LOGO_URL}" style="height:34px;width:auto;" alt="SIIMAG">
          </td>
          <td align="right">
            <span style="background:#EEF2FA;border:1px solid #ccd6f0;
                         padding:5px 14px;border-radius:20px;
                         font-size:10px;letter-spacing:1.5px;
                         color:{AZUL};{FM};font-weight:500;">{PERIODO_CORTO}</span>
          </td>
        </tr>
      </table>
    </td>
  </tr>

  <!-- HERO -->
  <tr>
    <td colspan="99" style="padding:28px 24px 20px;">
      <div style="font-size:11px;letter-spacing:3px;color:{AZUL};
                  text-transform:uppercase;margin-bottom:10px;
                  font-weight:700;{F}">Pulso Académico SIIMAG</div>
      <div style="font-size:38px;font-weight:800;line-height:1.0;
                  color:{AZUL};margin-bottom:10px;{F}">
        {len(indicadores)} indicadores clave
      </div>
      <div style="font-size:10px;color:#666;letter-spacing:1.5px;
                  text-transform:uppercase;margin-bottom:16px;{F}">
        Comparativo vs. {PERIODO_COMP_LABEL}
      </div>
      <div style="font-size:12px;color:#333;line-height:1.7;{F}">
        {texto_intro}
      </div>
    </td>
  </tr>

  <tr><td colspan="99" style="height:1px;background:{SEP};"></td></tr>

  {grilla(filas_indicadores)}

  <!-- FOOTER -->
  <tr>
    <td colspan="99" style="background:{AZUL};padding:14px 24px;">
      <table width="100%" cellpadding="0" cellspacing="0">
        <tr>
          <td style="font-size:11px;color:rgba(255,255,255,0.7);
                     font-weight:400;letter-spacing:0.5px;{F}">
            Sistema de Información, Inteligencia y Monitoreo · Academia Global
          </td>
          <td align="right">
            <a href="https://erp.agcollege.com.mx/#/login"
               style="font-size:10px;color:#ffffff;letter-spacing:2px;
                      text-transform:none;text-decoration:none;
                      font-weight:500;{F}">Si quieres saber sobre otros indicadores y su evolución, visita el SIIMAG</a>
          </td>
        </tr>
      </table>
    </td>
  </tr>

</table>
</td></tr>
</table>

</body>
</html>"""

print(f'✅ HTML generado — {len(html):,} caracteres')


✅ HTML generado — 8,497 caracteres


In [12]:
# ── 10. PREVIEW EN COLAB ─────────────────────────────────────────────────────
from IPython.display import HTML
HTML(html)

In [13]:
# ── 11. ENVÍO POR CORREO ──────────────────────────────────────────────────────
# El asunto lo genera Gemini con estilo Metro en la celda 5
# Sobre escribimos el asunto manualmente para este envío:
asunto_correo = 'Pulso Académico SIIMAG · Enero – Abril 2026'

lista_correos = [

    'claudio.pacheco@academiaglobal.mx'
]

REMITENTE    = 'siimag@academiaglobal.mx'
APP_PASSWORD = 'ealm gprb emme efez'

msg = MIMEMultipart('alternative')
msg['Subject'] = asunto_correo   # ← generado por Gemini con estilo Metro
msg['From']    = REMITENTE
msg['Bcc']     = ', '.join(lista_correos)
msg.attach(MIMEText(html, 'html'))

with smtplib.SMTP('smtp.gmail.com', 587) as server:
    server.starttls()
    server.login(REMITENTE, APP_PASSWORD)
    server.send_message(msg)

print(f'✅ Correo enviado a {len(lista_correos)} destinatarios')
print(f'📨 Asunto: {asunto_correo}')

✅ Correo enviado a 1 destinatarios
📨 Asunto: Pulso Académico SIIMAG · Enero – Abril 2026
